In [ ]:
import json
import os
import statistics
import time
from collections import Counter, defaultdict

from ortools.sat.python import cp_model
from generate import make_department
from groups import build_groups, session_demand

TRAIN_SEEDS = [1, 2, 3, 4]
CHECK_SEEDS = [101, 102]
WEEKS = ["sat", "sun"]

HARD_LIMIT = 60
ARMS = [30, 60, 120]
REPEATS = 4
TIME_COST = 2

W = {"gap": 5, "single_day": 4, "prof_pref": 3, "late_start": 2, "td_before_lecture": 2, "late": 1}

runs = len(ARMS) * len(TRAIN_SEEDS) * REPEATS
worst = (HARD_LIMIT + sum(ARMS) / len(ARMS)) * runs / 60
print(f"{runs} solves, at most about {worst:.0f} min")

In [ ]:
def department(seed, week):
    dept, _ = make_department("real", seed, vary=True, week=week)
    return dept

train = [(f"seed{s}", department(s, WEEKS[i % len(WEEKS)])) for i, s in enumerate(TRAIN_SEEDS)]
check = [(f"seed{s}", department(s, WEEKS[i % len(WEEKS)])) for i, s in enumerate(CHECK_SEEDS)]
for name, d in train + check:
    print(f"{name:8s} week={len(d['calendar']['days'])} days  {len(d['modules'])} modules  {len(d['professors'])} profs")

In [ ]:
class Problem:
    def __init__(self, dept):
        self.dept = dept
        self.groups = build_groups(dept)
        self.student_groups = [g for g in self.groups if g["kind"] == "group"]
        self.group_by_id = {g["id"]: g for g in self.groups}
        self.level_by_id = {l["id"]: l for l in dept["levels"]}
        self.days = dept["calendar"]["days"]
        self.hours = dept["rules"]["session_minutes"] / 60
        self.slots_per_day = len(dept["calendar"]["slots"])
        self.times = [(d, sl["id"]) for d in self.days for sl in dept["calendar"]["slots"]]
        self.time_index = {ts: t for t, ts in enumerate(self.times)}
        self.sessions = [{"id": f"s{i:03d}", "module": m, "kind": k, "group": g}
                         for i, (m, k, g) in enumerate(session_demand(dept, self.groups))]
        self.assign_profs()
        self.build()

    def assign_profs(self):
        dept = self.dept
        def candidates(s):
            return [(v["rank"], p["id"]) for p in dept["professors"] for v in p["voeux"]
                    if v["module"] == s["module"] and s["kind"] in v["kinds"]]
        am = cp_model.CpModel()
        y, cost = {}, []
        for s in self.sessions:
            for rank, p in candidates(s):
                y[s["id"], p] = am.NewBoolVar("")
                cost.append(rank * y[s["id"], p])
            am.AddExactlyOne(y[s["id"], p] for _, p in candidates(s))
        for p in dept["professors"]:
            mine = [v for (_, pid), v in y.items() if pid == p["id"]]
            if mine:
                am.Add(sum(mine) <= round(p["max_hours"] / self.hours))
        am.Minimize(sum(cost))
        sv = cp_model.CpSolver()
        sv.parameters.max_time_in_seconds = 10
        sv.Solve(am)
        for s in self.sessions:
            for _, p in candidates(s):
                if sv.Value(y[s["id"], p]):
                    s["prof"] = p

    def build(self):
        dept, times, ti, spd = self.dept, self.times, self.time_index, self.slots_per_day
        blocked = {(u["prof"], u["day"], u["slot"]) for u in dept["unavailability"]}
        room_types = dept["rules"]["room_types_for"]
        for s in self.sessions:
            g = self.group_by_id[s["group"]]
            s["rooms"] = [r["id"] for r in dept["rooms"]
                          if r["type"] in room_types[s["kind"]] and r["capacity"] >= g["size"]]
            s["times"] = [t for t, (d, sl) in enumerate(times)
                          if d in self.level_by_id[g["level"]]["days"] and (s["prof"], d, sl) not in blocked]

        m = self.model = cp_model.CpModel()
        self.x, self.when = x, when = {}, {}
        by_session, by_room, by_prof, by_group = (defaultdict(list) for _ in range(4))
        for s in self.sessions:
            for t in s["times"]:
                here = []
                for r in s["rooms"]:
                    v = m.NewBoolVar("")
                    x[s["id"], t, r] = v
                    here.append(v)
                    by_session[s["id"]].append(v)
                    by_room[t, r].append(v)
                when[s["id"], t] = m.NewBoolVar("")
                m.Add(when[s["id"], t] == sum(here))
                by_prof[t, s["prof"]].append(when[s["id"], t])
                by_group[t, s["group"]].append(when[s["id"], t])

        for s in self.sessions:
            m.AddExactlyOne(by_session[s["id"]])
        for vs in list(by_room.values()) + list(by_prof.values()):
            m.AddAtMostOne(vs)

        def group_vars(g, t):
            return by_group[t, g["id"]] + by_group[t, g["parent"]]

        for g in self.student_groups:
            for t in range(len(times)):
                if len(group_vars(g, t)) > 1:
                    m.AddAtMostOne(group_vars(g, t))
        for d in self.days:
            day_t = [t for t, (day, _) in enumerate(times) if day == d]
            for g in self.student_groups:
                vs = [v for t in day_t for v in group_vars(g, t)]
                if vs:
                    m.Add(sum(vs) <= dept["rules"]["max_sessions_per_day_group"])
            for p in dept["professors"]:
                vs = [v for t in day_t for v in by_prof[t, p["id"]]]
                if vs:
                    m.Add(sum(vs) <= dept["rules"]["max_sessions_per_day_prof"])

        self.dislikes = dislikes = set()
        for p in dept["professors"]:
            for d in p.get("dislikes", []):
                for day in (self.days if d["day"] == "all" else [d["day"]]):
                    for sl in (range(spd) if d["slot"] == "all" else [d["slot"]]):
                        dislikes.add((p["id"], day, sl))

        pen = self.penalties = []
        prof_of = {s["id"]: s["prof"] for s in self.sessions}
        for (sid, t), v in when.items():
            day, slot = times[t]
            if slot == spd - 1:
                pen.append(W["late"] * v)
            if (prof_of[sid], day, slot) in dislikes:
                pen.append(W["prof_pref"] * v)

        for s in self.sessions:
            g = self.group_by_id[s["group"]]
            if s["kind"] == "lecture" or g["kind"] != "group":
                continue
            for l in self.sessions:
                if l["kind"] != "lecture" or l["group"] != g["parent"] or l["module"] != s["module"]:
                    continue
                bad = m.NewBoolVar("")
                for d in self.days:
                    for i in range(spd):
                        a = when.get((s["id"], ti[d, i]))
                        later = [when[l["id"], ti[d, j]] for j in range(i + 1, spd) if (l["id"], ti[d, j]) in when]
                        if a is not None and later:
                            m.Add(bad >= a + sum(later) - 1)
                pen.append(W["td_before_lecture"] * bad)

        for g in self.student_groups:
            for d in self.level_by_id[g["level"]]["days"]:
                occ = []
                for i in range(spd):
                    o = m.NewBoolVar("")
                    m.Add(o == sum(group_vars(g, ti[d, i])))
                    occ.append(o)
                for i in range(1, spd - 1):
                    before, after, gap = m.NewBoolVar(""), m.NewBoolVar(""), m.NewBoolVar("")
                    for j in range(i):
                        m.Add(before >= occ[j])
                    for j in range(i + 1, spd):
                        m.Add(after >= occ[j])
                    m.Add(gap >= before + after - 1 - occ[i])
                    pen.append(W["gap"] * gap)
                busy = m.NewBoolVar("")
                for o in occ:
                    m.Add(busy >= o)
                single, late_start = m.NewBoolVar(""), m.NewBoolVar("")
                m.Add(single >= 2 * busy - sum(occ))
                m.Add(late_start >= busy - occ[0])
                pen += [W["single_day"] * single, W["late_start"] * late_start]

    def run(self, limit, seed):
        sv = cp_model.CpSolver()
        sv.parameters.max_time_in_seconds = limit
        sv.parameters.num_workers = os.cpu_count() or 1
        sv.parameters.random_seed = seed
        return sv, sv.Solve(self.model)

    def solve(self, soft_limit, seed=0):
        ok = (cp_model.OPTIMAL, cp_model.FEASIBLE)
        start = time.time()
        self.model.ClearHints()
        self.model.ClearObjective()
        sv, st = self.run(HARD_LIMIT, seed)
        if st not in ok:
            return {"status": "NO TIMETABLE", "time": round(time.time() - start, 1), "score": None, "sessions": []}
        for v in self.x.values():
            self.model.AddHint(v, sv.Value(v))
        self.model.Minimize(sum(self.penalties))
        sv2, st2 = self.run(soft_limit, seed)
        if st2 in ok:
            sv, st = sv2, st2
        score = round(sv.ObjectiveValue()) if st2 in ok else None
        return {"status": sv.StatusName(st), "time": round(time.time() - start, 1), "score": score,
                "sessions": self.read(sv)}

    def read(self, sv):
        by_id = {s["id"]: s for s in self.sessions}
        out = []
        for (sid, t, r), v in self.x.items():
            if sv.Value(v):
                s = by_id[sid]
                day, slot = self.times[t]
                out.append({"id": sid, "module": s["module"], "kind": s["kind"], "group": s["group"],
                            "prof": s["prof"], "room": r, "day": day, "slot": slot})
        return out

    def count_clashes(self, result):
        clashes = len(self.sessions) - len({r["id"] for r in result})
        for key in ("room", "prof"):
            c = Counter((r["day"], r["slot"], r[key]) for r in result)
            clashes += sum(n - 1 for n in c.values() if n > 1)
        for g in self.student_groups:
            c = Counter((r["day"], r["slot"]) for r in result if r["group"] in (g["id"], g["parent"]))
            clashes += sum(n - 1 for n in c.values() if n > 1)
        return clashes

    def evaluate(self, result):
        rep = Counter()
        for g in self.student_groups:
            mine = [r for r in result if r["group"] in (g["id"], g["parent"])]
            for d in self.days:
                used = sorted(r["slot"] for r in mine if r["day"] == d)
                if used:
                    rep["gaps"] += used[-1] - used[0] + 1 - len(used)
                    rep["late_start"] += used[0] > 0
                    rep["single_days"] += len(used) == 1
        rep["late"] = sum(r["slot"] == self.slots_per_day - 1 for r in result)
        rep["prof_pref_broken"] = sum((r["prof"], r["day"], r["slot"]) in self.dislikes for r in result)
        return dict(rep)

problems = [(name, Problem(d)) for name, d in train]
for name, p in problems:
    print(f"{name:8s} {len(p.sessions)} sessions, {len(p.x)} variables")

In [ ]:
def reward(p, out):
    if out["score"] is None:
        return -10_000
    return -out["score"] / len(p.sessions) * 100 - TIME_COST * out["time"] / 60

results = []
for rep in range(REPEATS):
    for arm in ARMS:
        for name, p in problems:
            out = p.solve(arm, seed=rep)
            results.append({"repeat": rep, "setting": arm, "department": name, "status": out["status"],
                            "time": out["time"], "score": out["score"],
                            "score_per_100": round(out["score"] / len(p.sessions) * 100, 1) if out["score"] is not None else None,
                            "clashes": p.count_clashes(out["sessions"]), "reward": round(reward(p, out), 2)})
            r = results[-1]
            print(f"rep {rep}  {arm:4d} s  {name:8s} {r['status']:12s} score {r['score']}  "
                  f"time {r['time']}  clashes {r['clashes']}  reward {r['reward']}")

In [ ]:
summary = []
for arm in ARMS:
    rs = [r for r in results if r["setting"] == arm]
    rw = [r["reward"] for r in rs]
    sc = [r["score_per_100"] for r in rs if r["score_per_100"] is not None]
    summary.append({"setting": arm, "runs": len(rs),
                    "mean_reward": round(statistics.mean(rw), 2), "spread": round(statistics.pstdev(rw), 2),
                    "worst_reward": min(rw), "mean_score_per_100": round(statistics.mean(sc), 1) if sc else None,
                    "mean_time": round(statistics.mean(r["time"] for r in rs), 1),
                    "clashes": sum(r["clashes"] for r in rs)})

print(f"{'limit':8s} {'runs':>4s} {'reward':>8s} {'spread':>7s} {'worst':>9s} {'score/100':>9s} {'time':>6s} {'clashes':>7s}")
for s in sorted(summary, key=lambda s: -s["mean_reward"]):
    print(f"{str(s['setting']) + ' s':8s} {s['runs']:4d} {s['mean_reward']:8.2f} {s['spread']:7.2f} {s['worst_reward']:9.2f} "
          f"{s['mean_score_per_100'] if s['mean_score_per_100'] is not None else '-':>9} {s['mean_time']:6.1f} {s['clashes']:7d}")

best = max(summary, key=lambda s: s["mean_reward"])["setting"]
print("\nbest SOFT_LIMIT:", best, "s")

In [ ]:
checks = []
for name, d in check:
    p = Problem(d)
    for rep in range(REPEATS):
        out = p.solve(best, seed=rep)
        checks.append({"department": name, "repeat": rep, "status": out["status"], "time": out["time"],
                       "score": out["score"], "clashes": p.count_clashes(out["sessions"]), **p.evaluate(out["sessions"])})
        print(checks[-1])

In [ ]:
with open("tuning_results.json", "w") as f:
    json.dump({"hard_limit": HARD_LIMIT, "best_soft_limit": best, "summary": summary, "runs": results, "check": checks}, f, indent=2)
print("saved tuning_results.json")
print(f"in solver.ipynb: TIME_LIMIT = {HARD_LIMIT}, SOFT_LIMIT = {best}")